# SARG Validator: Strict Local Bengali Evaluation
Metrics: Context Grounding, Faithfulness, Answer Relevancy, Instruction Following, Bengali Language Quality.


In [ ]:
!pip -q install -U google-genai pandas openpyxl tqdm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 62.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.0/260.0 kB 14.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.57.1 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.5 which is incompatible.


## 1. Configuration


In [ ]:
import os
import json
import time
import re
from pathlib import Path
from collections import deque
from datetime import datetime
import pandas as pd
from tqdm.auto import tqdm
from google import genai
from google.genai import types

# ----------------------------
# GEMINI CONFIGURATION
# ----------------------------
MODEL_NAME = "gemini-3.5-flash-lite"

# Limits from the provided Gemini console view
RPM_HARD = 15
RPM_TARGET = 12

TPM_HARD = 250_000
TPM_TARGET = 200_000

RPD_HARD = 500

# Evaluation settings
BATCH_SIZE = 4
TEMPERATURE = 0
MAX_RETRIES = 4

# Conservative output reservation per batch.
# Increase if your model regularly produces larger responses.
OUTPUT_TOKEN_RESERVE_PER_BATCH = 700

# Conservative token estimate for pre-send rate limiting.
# This is intentionally approximate. The safety buffer keeps us below TPM.
CHARS_PER_TOKEN_ESTIMATE = 3.5

# Files
CHECKPOINT_FILE = "sarg_gemini_checkpoint_strict_bengali_v2.csv"
FINAL_CSV = "sarg_gemini_final_report_strict_bengali_v2.csv"
FINAL_XLSX = "sarg_gemini_final_report_strict_bengali_v2.xlsx"

print({
    "model": MODEL_NAME,
    "batch_size": BATCH_SIZE,
    "rpm_target": RPM_TARGET,
    "tpm_target": TPM_TARGET,
    "rpd_hard": RPD_HARD
})


{'model': 'gemini-3.5-flash-lite', 'batch_size': 4, 'rpm_target': 12, 'tpm_target': 200000, 'rpd_hard': 500}


## 2. Add Gemini API key

The key is entered securely and is not written into the notebook output file.


In [ ]:
from getpass import getpass

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY") or getpass("Enter Gemini API key: ")
client = genai.Client(api_key=GEMINI_API_KEY)

print("Gemini client ready.")


Enter Gemini API key: ··········
Gemini client ready.


## 3. Upload dataset

Supported formats:
- `.jsonl` where each line is one row
- `.json` containing a list of rows


In [ ]:
from google.colab import files

uploaded = files.upload()
INPUT_FILE = next(iter(uploaded))
print("Uploaded:", INPUT_FILE)


Saving northbengal-sft-chat-bengali-clean.jsonl to northbengal-sft-chat-bengali-clean.jsonl
Uploaded: northbengal-sft-chat-bengali-clean.jsonl


## 4. Load dataset


In [ ]:
def load_dataset(path):
    suffix = Path(path).suffix.lower()

    if suffix == ".jsonl":
        rows = []
        with open(path, "r", encoding="utf-8") as f:
            for line_no, line in enumerate(f, start=1):
                line = line.strip()
                if not line:
                    continue
                try:
                    rows.append(json.loads(line))
                except json.JSONDecodeError as e:
                    rows.append({
                        "_load_error": f"Invalid JSONL at line {line_no}: {e}"
                    })
        return rows

    if suffix == ".json":
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        return data if isinstance(data, list) else [data]

    raise ValueError("Unsupported format. Upload a .json or .jsonl file.")

raw_rows = load_dataset(INPUT_FILE)
print("Rows loaded:", len(raw_rows))


Rows loaded: 1999


## 5. Extract Metadata Only as Context

Expected dataset structure:

```text
messages
├── system    -> Sahayak AI instructions + metadata
├── user      -> question
└── assistant -> answer
```

Only the metadata block from the system message is extracted and used as context.

Required metadata:

- জেলা (District)
- শ্রেণী / শ্রেণি (Grade/Class)
- বিষয় / বিষয় (Subject)
- পাঠ্য অধ্যায় / পাঠ্য অধ্যায় / বিষয়বস্তু / Topic (Chapter/Topic)

No other system instructions are passed to the LLM judge.


In [ ]:
METADATA_FIELDS = [
    ("জেলা", ["জেলা"]),
    ("শ্রেণী", ["শ্রেণী", "শ্রেণি"]),
    ("বিষয়", ["বিষয়", "বিষয়"]),
    ("পাঠ্য অধ্যায়", [
        "পাঠ্য অধ্যায়",
        "পাঠ্য অধ্যায়",
        "বিষয়বস্তু",
        "বিষয়বস্তু",
        "Topic",
        "Chapter",
    ]),
]

# Excel/XML-illegal control characters, excluding normal line breaks.
ILLEGAL_CONTROL_RE = re.compile(r"[\x00-\x08\x0B\x0C\x0E-\x1F]")

def extract_context(system_content):
    """Extract only metadata from the system message, never the full instructions."""
    if not isinstance(system_content, str):
        return ""

    # Prefer the explicit metadata block when present.
    metadata_match = re.search(
        r"(?:\[|\n)\s*[^]\n]*metadata\s*:\s*(.*?)(?:\]|\n\s*\n|$)",
        system_content,
        flags=re.IGNORECASE | re.DOTALL,
    )

    search_text = metadata_match.group(1) if metadata_match else system_content

    extracted = []

    for output_name, aliases in METADATA_FIELDS:
        value = None

        for alias in aliases:
            # Supports pipe-separated metadata:
            # জেলা: দার্জিলিং | শ্রেণী: একাদশ | বিষয়: অর্থনীতি
            pattern = re.compile(
                rf"{re.escape(alias)}\s*:\s*(.*?)(?=\s*\||\s*$|\n)",
                flags=re.IGNORECASE,
            )
            match = pattern.search(search_text)

            if match and match.group(1).strip():
                value = match.group(1).strip()
                break

            # Supports bullet/line metadata:
            # - জেলা: দার্জিলিং
            pattern = re.compile(
                rf"(?m)^\s*-\s*{re.escape(alias)}\s*:\s*(.+?)\s*$",
                flags=re.IGNORECASE,
            )
            match = pattern.search(search_text)

            if match and match.group(1).strip():
                value = match.group(1).strip()
                break

        if value:
            extracted.append(f"{output_name}: {value}")

    return "\n".join(extracted)


def prevalidate_answer_text(answer):
    """Detect objective text corruption before LLM evaluation."""
    issues = []

    if not isinstance(answer, str):
        return ["answer is not a string"]

    if ILLEGAL_CONTROL_RE.search(answer):
        issues.append("illegal control character")

    if re.search(r"\$\s*[\t\r\f\v]+", answer):
        issues.append("corrupted math expression after dollar sign")

    if answer.count("$") % 2 != 0:
        issues.append("unbalanced dollar math delimiter")

    return issues


def parse_row(raw_row, row_id):
    result = {
        "row_id": row_id,
        "context": "",
        "question": "",
        "answer": "",
        "schema_valid": False,
        "schema_issues": "",
        "text_quality_issues": ""
    }

    if not isinstance(raw_row, dict):
        result["schema_issues"] = "row is not a JSON object"
        return result

    if raw_row.get("_load_error"):
        result["schema_issues"] = raw_row["_load_error"]
        return result

    messages = raw_row.get("messages")
    if not isinstance(messages, list):
        result["schema_issues"] = "messages is missing or not a list"
        return result

    role_content = {}
    for msg in messages:
        if isinstance(msg, dict):
            role = msg.get("role")
            content = msg.get("content")
            if role in {"system", "user", "assistant"} and isinstance(content, str):
                role_content[role] = content

    system_content = role_content.get("system", "")
    result["context"] = extract_context(system_content)
    result["question"] = role_content.get("user", "").strip()
    result["answer"] = role_content.get("assistant", "").strip()

    issues = []

    if not system_content:
        issues.append("missing system message")
    elif not result["context"]:
        issues.append("metadata missing or could not be extracted")

    if not result["question"]:
        issues.append("missing user question")

    if not result["answer"]:
        issues.append("missing assistant answer")

    result["schema_issues"] = "; ".join(issues)
    result["schema_valid"] = len(issues) == 0
    result["text_quality_issues"] = "; ".join(
        prevalidate_answer_text(result["answer"])
    )

    return result


parsed_rows = [parse_row(row, i) for i, row in enumerate(raw_rows)]
df = pd.DataFrame(parsed_rows)

print("Total rows:", len(df))
print("Schema-valid rows:", int(df["schema_valid"].sum()))
print("Schema-invalid rows:", int((~df["schema_valid"]).sum()))
print("Rows with pre-validation text issues:",
      int(df["text_quality_issues"].astype(bool).sum()))

print("\nExtracted metadata context:")
display(df[["row_id", "context"]].head(10))

print("\nPre-validation text issues:")
display(df[["row_id", "text_quality_issues"]].head(10))

print("\nSchema check:")
display(df[["row_id", "schema_valid", "schema_issues"]].head(10))


Total rows: 1999
Schema-valid rows: 1999
Schema-invalid rows: 0
Rows with pre-validation text issues: 24

Extracted metadata context:


,row_id,context
0,0,জেলা: দার্জিলিং\nশ্রেণী: একাদশ\nবিষয়: অর্থনীত...
1,1,জেলা: কালিম্পং\nশ্রেণী: ৯ম\nবিষয়: বিজ্ঞান (Sc...
2,2,জেলা: কালিম্পং\nশ্রেণী: ৪র্থ\nবিষয়: গণিত (Mat...
3,3,জেলা: কোচবিহার\nশ্রেণী: ১ম\nবিষয়: গণিত (Mathe...
4,4,জেলা: কালিম্পং\nশ্রেণী: ১০ম\nবিষয়: গণিত (Math...
5,5,জেলা: জলপাইগুড়ি\nশ্রেণী: দ্বাদশ\nবিষয়: রসায়...
6,6,জেলা: দক্ষিণ দিনাজপুর\nশ্রেণী: একাদশ\nবিষয়: জ...
7,7,জেলা: মালদা\nশ্রেণী: ৬ষ্ঠ\nবিষয়: গণিত (Mathem...
8,8,জেলা: মালদা\nশ্রেণী: ৮ম\nবিষয়: গণিত (Mathemat...
9,9,জেলা: আলিপুরদুয়ার\nশ্রেণী: ৬ষ্ঠ\nবিষয়: বিজ্ঞ...



Pre-validation text issues:


,row_id,text_quality_issues
0,0,
1,1,
2,2,
3,3,
4,4,
5,5,
6,6,
7,7,
8,8,
9,9,



Schema check:


,row_id,schema_valid,schema_issues
0,0,True,
1,1,True,
2,2,True,
3,3,True,
4,4,True,
5,5,True,
6,6,True,
7,7,True,
8,8,True,
9,9,True,


In [ ]:
pd.set_option("display.max_colwidth", None)

display(
    df[[
        "row_id",
        "context",
        "question",
        "answer"
    ]]
)

,row_id,context,question,answer
0,0,জেলা: দার্জিলিং\nশ্রেণী: একাদশ\nবিষয়: অর্থনীতি (Economics)\nপাঠ্য অধ্যায়: Adjustment of Class Limits in Frequency Distribution,একাদশ শ্রেণির অর্থনীতির ফ্রিকোয়েন্সি ডিস্ট্রিবিউশনে ক্লাস লিমিট অ্যাডজাস্টমেন্ট বা শ্রেণি সীমানার সংশোধন নিয়ে একটি অনুশীলনী প্রশ্ন দাও।,"প্রশ্ন: দার্জিলিংয়ের চা পাতা তোলার মরসুমে, রেঞ্জিং বিভিন্ন শ্রমিকের সংগৃহীত কাঁচা চা পাতার দৈনিক ওজন (কিলোগ্রামে) নথিভুক্ত করল। দৈনিক চা সংগ্রহের বিচ্ছিন্ন ফ্রিকোয়েন্সি বা পরিসংখ্যা বিভাজনটি দেওয়া হল: ১০-১৯, ২০-২৯, ৩০-৩৯ এবং ৪০-৪৯। শ্রেণি সীমানা সংশোধন করে এই বিচ্ছিন্ন পরিসংখ্যা বিভাজনটিকে একটি অবিচ্ছিন্ন পরিসংখ্যা বিভাজনে রূপান্তর করো এবং তৃতীয় শ্রেণি অন্তরালের সংশোধিত শ্রেণি মধ্যক গণনা করো। সমাধান: ধাপ ১: একটি শ্রেণির ঊর্ধ্বসীমা এবং পরবর্তী শ্রেণির নিম্নসীমার মধ্যবর্তী ব্যবধান চিহ্নিত করো। ব্যবধান (d) = ২০ - ১৯ = ১। ধাপ ২: ব্যবধানকে ২ দিয়ে ভাগ করে সংশোধনের উপাদান বা অ্যাডজাস্টমেন্ট ফ্যাক্টর নির্ণয় করো। অ্যাডজাস্টমেন্ট ফ্যাক্টর = ১ / ২ = ০.৫। ধাপ ৩: শ্রেণিগুলিকে অবিচ্ছিন্ন করার জন্য প্রতিটি নিম্নশ্রেণি সীমা থেকে ০.৫ বিয়োগ করো এবং প্রতিটি ঊর্ধ্বশ্রেণি সীমার সাথে ০.৫ যোগ করো। - প্রথম শ্রেণি: ১০ - ০.৫ থেকে ১৯ + ০.৫ = ৯.৫ - ১৯.৫। - দ্বিতীয় শ্রেণি: ২০ - ০.৫ থেকে ২৯ + ০.৫ = ১৯.৫ - ২৯.৫। - তৃতীয় শ্রেণি: ৩০ - ০.৫ থেকে ৩৯ + ০.৫ = ২৯.৫ - ৩৯.৫। - চতুর্থ শ্রেণি: ৪০ - ০.৫ থেকে ৪৯ + ০.৫ = ৩৯.৫ - ৪৯.৫। ধাপ ৪: তৃতীয় শ্রেণি অন্তরালের (২৯.৫ - ৩৯.৫) সংশোধিত শ্রেণি মধ্যক গণনা করো। শ্রেণি মধ্যক = (নিম্নসীমা + ঊর্ধ্বসীমা) / ২। শ্রেণি মধ্যক = (২৯.৫ + ৩৯.৫) / ২ = ৬৯.০ / ২ = ৩৪.৫। চূড়ান্ত উত্তর: অবিচ্ছিন্ন শ্রেণিগুলি হল ৯.৫-১৯.৫, ১৯.৫-২৯.৫, ২৯.৫-৩৯.৫ এবং ৩৯.৫-৪৯.৫। তৃতীয় শ্রেণির সংশোধিত শ্রেণি মধ্যক হল ৩৪.৫।"
1,1,জেলা: কালিম্পং\nশ্রেণী: ৯ম\nবিষয়: বিজ্ঞান (Science)\nপাঠ্য অধ্যায়: Acceleration,নবম শ্রেণির বিজ্ঞানের জন্য ত্বরণের উপর একটি অনুশীলনী প্রশ্ন দিন।,"প্রশ্ন: দেওলো পাহাড়ে একটি পরীক্ষার সময়, লাকপা একটি গাড়িকে ঢাল বেয়ে নামতে দেখল। গাড়িটির বেগ ৫.০ সেকেন্ডে ২.০ মিটার/সেকেন্ড থেকে বেড়ে ১২.০ মিটার/সেকেন্ড হয়। গাড়িটির ত্বরণ নির্ণয় কর। সমাধান: ধাপ ১: পর্যবেক্ষণ থেকে প্রদত্ত মানগুলি চিহ্নিত কর। প্রাথমিক বেগ (u) = ২.০ মিটার/সেকেন্ড, অন্তিম বেগ (v) = ১২.০ মিটার/সেকেন্ড, সময় (t) = ৫.০ সেকেন্ড। ধাপ ২: ত্বরণের সূত্রটি মনে কর: a = (v - u) / t। ধাপ ৩: সূত্রটিতে মানগুলি বসাও: a = (১২.০ - ২.০) / ৫.০। ধাপ ৪: লব গণনা কর: ১২.০ - ২.০ = ১০.০ মিটার/সেকেন্ড। ধাপ ৫: সময় দ্বারা ভাগ কর: ১০.০ / ৫.০ = ২.০ মিটার/সেকেন্ড²। চূড়ান্ত উত্তর: গাড়িটির ত্বরণ হল ২.০ মিটার/সেকেন্ড²।"
2,2,জেলা: কালিম্পং\nশ্রেণী: ৪র্থ\nবিষয়: গণিত (Mathematics)\nপাঠ্য অধ্যায়: Tally marks and reading a data table,চতুর্থ শ্রেণির গণিতের জন্য ট্যালিস বা ট্যালি চিহ্ন এবং ডেটা টেবিল বা তথ্য সারণি পড়ার ধারণাটি বুঝিয়ে দিন।,"ধারণার ব্যাখ্যা: চতুর্থ শ্রেণির গণিতে, আমাদের চারপাশের জিনিসপত্র গোনার সময় ট্যালি চিহ্ন খুব দ্রুত তথ্য গুনে নথিভুক্ত করতে সাহায্য করে। আমরা যখনই কোনো একটি জিনিস গুনি, তখন একটি লম্বালম্বি দাগ টানি। প্রতি পঞ্চম জিনিসটির জন্য, আমরা প্রথম চারটি দাগের ওপর একটি তির্যক দাগ টানি যাতে পাঁচটি দাগের একটি আঁটি তৈরি হয় (চারটি লম্বালম্বি দাগ এবং তার ওপর একটি কাটা দাগ)। ডেটা টেবিল বা তথ্য সারণি পড়া মানে সহজেই সংখ্যা খুঁজে পেতে এবং তুলনা করতে সারি ও কলামগুলো দেখা। কালিম্পংয়ের পাহাড়ের একটি স্কুলের কথা ভাবা যাক, যেখানে সোনম এবং লাকপা তাদের শিক্ষককে একটি সাংস্কৃতিক প্রদর্শনীর জন্য জিনিসপত্র সাজাতে সাহায্য করছে। তারা শিক্ষার্থীদের নিয়ে আসা বিভিন্ন স্থানীয় জিনিস গুনে দেখল: বড় এলাচ: ১২, আদা: ৮, ফুলের টব: ১৫, থাঙ্কা চিত্রশিল্প: ৫। প্রশ্ন: আমরা কীভাবে ট্যালি চিহ্নের সাহায্যে বড় এলাচের সংখ্যা প্রকাশ করব এবং তথ্য সারণিতে কোন জিনিসটির সংখ্যা সবচেয়ে বেশি? সমাধান: ধাপ ১: বড় এলাচের সংখ্যা হলো ১২। আমরা ট্যালি চিহ্নে ১২-কে পাঁচটি করে দুটি আঁটি এবং দুটি একক দাগ দিয়ে প্রকাশ করি: |||| |||| ||। ধাপ ২: তথ্য সারণির সমস্ত জিনিসগুলো দেখে (বড় এলাচ: ১২, আদা: ৮, ফুলের টব: ১৫, থাঙ্কা চিত্রশিল্প: ৫), আমরা সংখ্যাগুলোর তুলনা করি। ধাপ ৩: সবচেয়ে বেশি সংখ্যা হলো ১৫, যা ফুলের টবের ক্ষেত্রে রয়েছে। চূড়ান্ত উত্তর: বড় এলাচকে দুটি আঁটি ও দুটি একক দাগ দিয়ে ট্যালি করা হয় এবং ফুলের টবের সংখ্যা সবচেয়ে বেশি।"
3,3,জেলা: কোচবিহার\nশ্রেণী: ১ম\nবিষয়: গণিত (Mathematics)\nপাঠ্য অধ্যায়: Continuing a simple repeating pattern,প্রথম শ্রেণির গণিতের জন্য স

## 6. Strict evaluation prompt
Adds hard scoring constraints, Bengali language quality scoring, and pre-validation text issue awareness.


In [ ]:
EVALUATOR_INSTRUCTION = """প্রতিটি আইটেম স্বাধীনভাবে এবং কঠোরভাবে মূল্যায়ন করো। শুধুমাত্র বৈধ JSON ফেরত দাও।

সব স্কোর পূর্ণসংখ্যা ০–৪:
g = context groundedness: উত্তরটি প্রাসঙ্গিক context যথাযথভাবে ব্যবহার বা সম্মান করে কি না; সব field ব্যবহার করা বাধ্যতামূলক নয়।
f = faithfulness: উত্তরটি context-এর বিরোধিতা করে বা অসমর্থিত স্থানীয়/বিষয়ভিত্তিক তথ্য তৈরি করে কি না।
r = answer relevancy: উত্তরটি প্রশ্নের সরাসরি ও যথেষ্ট উত্তর দেয় কি না।
i = instruction following: উত্তরটি অনুরোধকৃত কাজ, বিন্যাস, শ্রেণি/বিষয়ের সীমাবদ্ধতা ও নির্দেশনা অনুসরণ করে কি না।
b = bengali language quality: পশ্চিমবঙ্গের বাংলা ভাষাভাষী শিক্ষার্থীদের জন্য বাংলা ভাষার শুদ্ধতা, স্বাভাবিকতা, সাবলীলতা, শব্দচয়ন ও বাক্যগঠন।

বাংলা কঠোরভাবে পরীক্ষা করো:
- বাংলা স্বাভাবিক, শুদ্ধ, সাবলীল এবং পশ্চিমবঙ্গে প্রচলিত হওয়া উচিত।
- প্রাসঙ্গিক হলে উত্তরটি প্রদত্ত অঞ্চল এবং rural/semi_rural প্রেক্ষাপটের সঙ্গে সামঞ্জস্যপূর্ণ হবে; কৃত্রিম আঞ্চলিক শব্দ ব্যবহার করা যাবে না।
- ভুল ব্যাকরণ, অস্বাভাবিক বাক্যগঠন, আক্ষরিক অনুবাদ, যান্ত্রিক বাংলা, অপ্রচলিত শব্দচয়ন, অস্বাভাবিক সম্বোধন এবং উপযুক্ত বাংলা থাকা সত্ত্বেও অপ্রয়োজনীয় ইংরেজি/হিন্দি/বিদেশি শব্দের জন্য নম্বর কমাও।
- প্রয়োজনীয় আন্তর্জাতিক প্রতীক, সূত্র, একক বা প্রতিষ্ঠিত বৈজ্ঞানিক notation-কে ভুল হিসেবে গণ্য করবে না।
- স্থানীয় উদাহরণ অবশ্যই প্রাসঙ্গিক ও স্বাভাবিক হতে হবে; শুধু “local” দেখানোর জন্য যোগ করা হলে নম্বর কমাও।

স্কোরিং নিয়ম:
- স্পষ্ট ত্রুটি থাকলে সংশ্লিষ্ট স্কোর ৪ হতে পারবে না।
- বাংলা বা ভাষাগত ত্রুটি থাকলে b = ৪ হবে না; instruction অনুসরণে প্রভাব ফেললে i = ৪-ও হবে না।
- corrupted text, illegal control character বা malformed math থাকলে প্রভাবিত quality score ৪ হবে না।
- ৪ = কোনো স্পষ্ট সমস্যা নেই; ৩ = সামান্য সমস্যা; ২ = লক্ষ্যণীয় সমস্যা; ১ = বড় সমস্যা; ০ = ব্যর্থ।

reason নিয়ম:
- পাঁচটি স্কোরই ৪ হলে reason খালি string ("") হবে। কোনো প্রশংসা, ব্যাখ্যা, সারাংশ বা অন্য কোনো লেখা তৈরি করবে না।
- একটি বা একাধিক স্কোর ৪-এর কম হলে শুধুমাত্র স্কোর কমার গুরুত্বপূর্ণ কারণগুলো সংক্ষেপে reason-এ উল্লেখ করবে।

নিচের আইটেমগুলো মূল্যায়ন করো:
"""


def build_batch_prompt(batch):
    parts = [EVALUATOR_INSTRUCTION]

    for item in batch:
        parts.append(
            f"\nID:{item['row_id']}\n"
            f"C:{item['context']}\n"
            f"Q:{item['question']}\n"
            f"A:{item['answer']}\n"
            f"PREVALIDATION:{item.get('text_quality_issues', '') or 'none'}\n"
        )

    return "".join(parts)


## 7. Structured output schema

Gemini is instructed to return a predictable JSON structure.


In [ ]:
EVAL_SCHEMA = {
    "type": "object",
    "properties": {
        "results": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "id": {"type": "integer"},
                    "g": {"type": "integer", "minimum": 0, "maximum": 4},
                    "f": {"type": "integer", "minimum": 0, "maximum": 4},
                    "r": {"type": "integer", "minimum": 0, "maximum": 4},
                    "i": {"type": "integer", "minimum": 0, "maximum": 4},
                    "b": {"type": "integer", "minimum": 0, "maximum": 4},
                    "reason": {"type": "string"}
                },
                "required": ["id", "g", "f", "r", "i", "b", "reason"]
            }
        }
    },
    "required": ["results"]
}


## 8. Token estimator and rolling rate limiter

The limiter tracks:
- Requests in the last 60 seconds
- Estimated tokens in the last 60 seconds
- Total requests made during the run

The code targets 12 RPM and 200K TPM even though the displayed hard limits are 15 RPM and 250K TPM.


In [ ]:
def estimate_tokens(text):
    return max(1, int(len(text) / CHARS_PER_TOKEN_ESTIMATE))


class RollingRateLimiter:
    def __init__(
        self,
        rpm_target=RPM_TARGET,
        tpm_target=TPM_TARGET,
        rpd_hard=RPD_HARD
    ):
        self.rpm_target = rpm_target
        self.tpm_target = tpm_target
        self.rpd_hard = rpd_hard

        self.request_events = deque()  # (timestamp, token_estimate)
        self.daily_requests = 0

    def _cleanup(self):
        now = time.monotonic()
        while self.request_events and now - self.request_events[0][0] >= 60:
            self.request_events.popleft()

    def current_rpm(self):
        self._cleanup()
        return len(self.request_events)

    def current_tpm(self):
        self._cleanup()
        return sum(tokens for _, tokens in self.request_events)

    def wait_until_safe(self, estimated_tokens):
        if self.daily_requests >= self.rpd_hard:
            raise RuntimeError(
                f"RPD hard limit reached: {self.daily_requests}/{self.rpd_hard}"
            )

        while True:
            self._cleanup()

            requests_ok = len(self.request_events) < self.rpm_target
            tokens_ok = (
                sum(tokens for _, tokens in self.request_events) + estimated_tokens
                <= self.tpm_target
            )

            if requests_ok and tokens_ok:
                return

            now = time.monotonic()
            waits = []

            if not requests_ok and self.request_events:
                waits.append(60 - (now - self.request_events[0][0]))

            if not tokens_ok and self.request_events:
                running = sum(tokens for _, tokens in self.request_events)
                cumulative = running

                for ts, tokens in self.request_events:
                    if cumulative + estimated_tokens <= self.tpm_target:
                        waits.append(60 - (now - ts))
                        break
                    cumulative -= tokens
                else:
                    waits.append(60 - (now - self.request_events[0][0]))

            sleep_for = max(0.5, min(w for w in waits if w > 0) if waits else 1.0)

            print(
                f"Rate limiter waiting {sleep_for:.1f}s | "
                f"RPM={self.current_rpm()}/{self.rpm_target} | "
                f"TPM≈{self.current_tpm():,}/{self.tpm_target:,}"
            )
            time.sleep(sleep_for)

    def record(self, estimated_tokens):
        self._cleanup()
        self.request_events.append((time.monotonic(), estimated_tokens))
        self.daily_requests += 1


limiter = RollingRateLimiter()
print("Rate limiter ready.")


Rate limiter ready.


## 9. Gemini batch evaluator

If a batch fails, it retries with exponential backoff.

If Gemini returns:
- invalid JSON
- missing rows
- duplicate IDs
- unexpected IDs

the batch is retried. After the retry limit, the batch is automatically split into smaller batches.


In [ ]:
def clamp_score(value):
    try:
        value = int(value)
        return max(0, min(4, value))
    except Exception:
        return None


def normalize_results(result_json, expected_ids):
    results = result_json.get("results", [])
    if not isinstance(results, list):
        raise ValueError("results is not a list")

    mapped = {}

    for item in results:
        row_id = item.get("id")
        if row_id in mapped:
            raise ValueError(f"duplicate result ID: {row_id}")

        mapped[row_id] = {
            "g": clamp_score(item.get("g")),
            "f": clamp_score(item.get("f")),
            "r": clamp_score(item.get("r")),
            "i": clamp_score(item.get("i")),
            "b": clamp_score(item.get("b")),
            "reason": str(item.get("reason", "")).strip()
        }

    missing = set(expected_ids) - set(mapped)
    extra = set(mapped) - set(expected_ids)

    if missing or extra:
        raise ValueError(
            f"ID mismatch | missing={sorted(missing)} extra={sorted(extra)}"
        )

    for row_id, scores in mapped.items():
        if None in [scores["g"], scores["f"], scores["r"], scores["i"], scores["b"]]:
            raise ValueError(f"invalid score returned for ID {row_id}")

    return mapped


def call_gemini(batch):
    prompt = build_batch_prompt(batch)

    estimated_input = estimate_tokens(prompt)
    estimated_total = estimated_input + OUTPUT_TOKEN_RESERVE_PER_BATCH

    limiter.wait_until_safe(estimated_total)

    last_error = None

    for attempt in range(MAX_RETRIES):
        try:
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt,
                config=types.GenerateContentConfig(
                    temperature=TEMPERATURE,
                    response_mime_type="application/json",
                    response_json_schema=EVAL_SCHEMA
                )
            )

            limiter.record(estimated_total)

            result_json = json.loads(response.text)
            expected_ids = [item["row_id"] for item in batch]
            return normalize_results(result_json, expected_ids)

        except Exception as e:
            last_error = e
            wait = min(30, 2 ** attempt)

            print(
                f"Batch {[x['row_id'] for x in batch]} failed "
                f"(attempt {attempt + 1}/{MAX_RETRIES}): {str(e)[:180]}"
            )

            if attempt < MAX_RETRIES - 1:
                time.sleep(wait)

    raise RuntimeError(str(last_error))


def evaluate_batch_adaptive(batch):
    try:
        return call_gemini(batch)

    except Exception as e:
        # Split only when there is more than one row.
        # This prevents one problematic/large batch from stopping the run.
        if len(batch) > 1:
            mid = len(batch) // 2
            print(
                f"Splitting batch {[x['row_id'] for x in batch]} "
                f"into {len(batch[:mid])} + {len(batch[mid:])}"
            )

            left = evaluate_batch_adaptive(batch[:mid])
            right = evaluate_batch_adaptive(batch[mid:])

            left.update(right)
            return left

        row_id = batch[0]["row_id"]
        return {
            row_id: {
                "g": None,
                "f": None,
                "r": None,
                "i": None,
                "b": None,
                "reason": f"EVALUATION_ERROR: {str(e)[:300]}"
            }
        }


## 10. Checkpoint helpers

The checkpoint stores completed rows. If the notebook stops, rerunning the notebook can continue from the remaining rows.


In [ ]:
def load_checkpoint(path=CHECKPOINT_FILE):
    if not Path(path).exists():
        return pd.DataFrame()

    checkpoint = pd.read_csv(path)

    if "row_id" not in checkpoint.columns:
        return pd.DataFrame()

    return checkpoint


def save_checkpoint(result_df, path=CHECKPOINT_FILE):
    result_df.to_csv(path, index=False)


checkpoint_df = load_checkpoint()
completed_ids = set(checkpoint_df["row_id"].astype(int)) if not checkpoint_df.empty else set()

print("Checkpoint rows:", len(completed_ids))
print("Rows remaining:", len(df) - len(completed_ids))


Checkpoint rows: 0
Rows remaining: 1999


## 11. Run evaluation

Schema-invalid rows are written directly to the output with `schema_failed`.

Valid rows are evaluated in batches of 4.


In [ ]:
# Create immediate results for invalid rows.
invalid_results = []

for _, row in df[~df["schema_valid"]].iterrows():
    invalid_results.append({
        "row_id": int(row["row_id"]),
        "schema_valid": False,
        "schema_issues": row["schema_issues"],
        "context_grounding": None,
        "faithfulness": None,
        "answer_relevancy": None,
        "instruction_following": None,
        "bengali_language_quality": None,
        "overall_score": None,
        "passed": False,
        "reason": "schema_failed"
    })


valid_rows = df[df["schema_valid"]].copy()

# Exclude rows already completed in checkpoint.
pending_rows = valid_rows[
    ~valid_rows["row_id"].isin(completed_ids)
].to_dict("records")

# If checkpoint exists, use it as completed results.
all_results = checkpoint_df.to_dict("records") if not checkpoint_df.empty else []

# Ensure invalid rows are present.
existing_ids = {int(x["row_id"]) for x in all_results if "row_id" in x}
for item in invalid_results:
    if item["row_id"] not in existing_ids:
        all_results.append(item)

pending_batches = [
    pending_rows[i:i + BATCH_SIZE]
    for i in range(0, len(pending_rows), BATCH_SIZE)
]

print("Valid pending rows:", len(pending_rows))
print("Batches to send:", len(pending_batches))

for batch in tqdm(pending_batches, desc="Evaluating"):
    batch_scores = evaluate_batch_adaptive(batch)

    for item in batch:
        row_id = item["row_id"]
        score = batch_scores[row_id]

        g = score["g"]
        f = score["f"]
        r = score["r"]
        i = score["i"]
        b = score["b"]

        numeric_scores = [x for x in [g, f, r, i, b] if x is not None]
        overall = round(sum(numeric_scores) / 5, 2) if len(numeric_scores) == 5 else None

        # Conservative pass rule.
        passed = (
            overall is not None
            and overall >= 3.0
            and min(numeric_scores) >= 2
        )

        all_results.append({
            "row_id": row_id,
            "schema_valid": True,
            "schema_issues": "",
            "context_grounding": g,
            "faithfulness": f,
            "answer_relevancy": r,
            "instruction_following": i,
            "bengali_language_quality": b,
            "overall_score": overall,
            "passed": passed,
            "reason": score["reason"]
        })

    # Save after every batch.
    checkpoint_out = (
        pd.DataFrame(all_results)
        .drop_duplicates(subset=["row_id"], keep="last")
        .sort_values("row_id")
    )
    save_checkpoint(checkpoint_out)

print("Evaluation complete.")


Valid pending rows: 1999
Batches to send: 500


Evaluating:   0%|          | 0/500 [00:00<?, ?it/s]

Rate limiter waiting 46.5s | RPM=12/12 | TPM≈32,000/200,000
Rate limiter waiting 0.5s | RPM=12/12 | TPM≈31,762/200,000
Rate limiter waiting 46.2s | RPM=12/12 | TPM≈31,736/200,000
Rate limiter waiting 1.9s | RPM=12/12 | TPM≈32,313/200,000
Rate limiter waiting 5.3s | RPM=12/12 | TPM≈31,942/200,000
Rate limiter waiting 6.6s | RPM=12/12 | TPM≈32,667/200,000
Rate limiter waiting 0.5s | RPM=12/12 | TPM≈32,311/200,000
Rate limiter waiting 7.7s | RPM=12/12 | TPM≈32,118/200,000
Rate limiter waiting 4.2s | RPM=12/12 | TPM≈32,346/200,000
Rate limiter waiting 0.5s | RPM=12/12 | TPM≈32,052/200,000
Rate limiter waiting 6.4s | RPM=12/12 | TPM≈32,536/200,000
Rate limiter waiting 13.2s | RPM=12/12 | TPM≈32,560/200,000
Rate limiter waiting 21.4s | RPM=12/12 | TPM≈32,449/200,000
Rate limiter waiting 2.9s | RPM=12/12 | TPM≈32,971/200,000
Rate limiter waiting 0.6s | RPM=12/12 | TPM≈33,251/200,000
Rate limiter waiting 6.4s | RPM=12/12 | TPM≈32,682/200,000
Rate limiter waiting 13.3s | RPM=12/12 | TPM≈32,909/

## 12. Merge results with original extracted fields and export final report


In [ ]:
results_df = (
    pd.DataFrame(all_results)
    .drop_duplicates(subset=["row_id"], keep="last")
    .sort_values("row_id")
)

base_columns = [
    "row_id",
    "context",
    "question",
    "answer",
    "schema_valid",
    "schema_issues",
    "text_quality_issues"
]

final_df = df[base_columns].merge(
    results_df.drop(
        columns=["schema_valid", "schema_issues"],
        errors="ignore"
    ),
    on="row_id",
    how="left"
)

ordered_columns = [
    "row_id",
    "schema_valid",
    "schema_issues",
    "text_quality_issues",
    "context_grounding",
    "faithfulness",
    "answer_relevancy",
    "instruction_following",
    "bengali_language_quality",
    "overall_score",
    "passed",
    "reason",
    "question",
    "answer",
    "context"
]

final_df = final_df[[c for c in ordered_columns if c in final_df.columns]]

# Remove illegal control characters only for Excel/CSV export.
def clean_export_value(value):
    if isinstance(value, str):
        return ILLEGAL_CONTROL_RE.sub("", value)
    return value

export_df = final_df.map(clean_export_value)

export_df.to_csv(FINAL_CSV, index=False)
export_df.to_excel(FINAL_XLSX, index=False)

print("Saved:")
print(FINAL_CSV)
print(FINAL_XLSX)

display(final_df.head())


Saved:
sarg_gemini_final_report_strict_bengali_v2.csv
sarg_gemini_final_report_strict_bengali_v2.xlsx


,row_id,schema_valid,schema_issues,text_quality_issues,context_grounding,faithfulness,answer_relevancy,instruction_following,bengali_language_quality,overall_score,passed,reason,question,answer,context
0,0,True,,,4.0,4.0,4.0,4.0,4.0,4.0,True,,একাদশ শ্রেণির অর্থনীতির ফ্রিকোয়েন্সি ডিস্ট্রিবিউশনে ক্লাস লিমিট অ্যাডজাস্টমেন্ট বা শ্রেণি সীমানার সংশোধন নিয়ে একটি অনুশীলনী প্রশ্ন দাও।,"প্রশ্ন: দার্জিলিংয়ের চা পাতা তোলার মরসুমে, রেঞ্জিং বিভিন্ন শ্রমিকের সংগৃহীত কাঁচা চা পাতার দৈনিক ওজন (কিলোগ্রামে) নথিভুক্ত করল। দৈনিক চা সংগ্রহের বিচ্ছিন্ন ফ্রিকোয়েন্সি বা পরিসংখ্যা বিভাজনটি দেওয়া হল: ১০-১৯, ২০-২৯, ৩০-৩৯ এবং ৪০-৪৯। শ্রেণি সীমানা সংশোধন করে এই বিচ্ছিন্ন পরিসংখ্যা বিভাজনটিকে একটি অবিচ্ছিন্ন পরিসংখ্যা বিভাজনে রূপান্তর করো এবং তৃতীয় শ্রেণি অন্তরালের সংশোধিত শ্রেণি মধ্যক গণনা করো। সমাধান: ধাপ ১: একটি শ্রেণির ঊর্ধ্বসীমা এবং পরবর্তী শ্রেণির নিম্নসীমার মধ্যবর্তী ব্যবধান চিহ্নিত করো। ব্যবধান (d) = ২০ - ১৯ = ১। ধাপ ২: ব্যবধানকে ২ দিয়ে ভাগ করে সংশোধনের উপাদান বা অ্যাডজাস্টমেন্ট ফ্যাক্টর নির্ণয় করো। অ্যাডজাস্টমেন্ট ফ্যাক্টর = ১ / ২ = ০.৫। ধাপ ৩: শ্রেণিগুলিকে অবিচ্ছিন্ন করার জন্য প্রতিটি নিম্নশ্রেণি সীমা থেকে ০.৫ বিয়োগ করো এবং প্রতিটি ঊর্ধ্বশ্রেণি সীমার সাথে ০.৫ যোগ করো। - প্রথম শ্রেণি: ১০ - ০.৫ থেকে ১৯ + ০.৫ = ৯.৫ - ১৯.৫। - দ্বিতীয় শ্রেণি: ২০ - ০.৫ থেকে ২৯ + ০.৫ = ১৯.৫ - ২৯.৫। - তৃতীয় শ্রেণি: ৩০ - ০.৫ থেকে ৩৯ + ০.৫ = ২৯.৫ - ৩৯.৫। - চতুর্থ শ্রেণি: ৪০ - ০.৫ থেকে ৪৯ + ০.৫ = ৩৯.৫ - ৪৯.৫। ধাপ ৪: তৃতীয় শ্রেণি অন্তরালের (২৯.৫ - ৩৯.৫) সংশোধিত শ্রেণি মধ্যক গণনা করো। শ্রেণি মধ্যক = (নিম্নসীমা + ঊর্ধ্বসীমা) / ২। শ্রেণি মধ্যক = (২৯.৫ + ৩৯.৫) / ২ = ৬৯.০ / ২ = ৩৪.৫। চূড়ান্ত উত্তর: অবিচ্ছিন্ন শ্রেণিগুলি হল ৯.৫-১৯.৫, ১৯.৫-২৯.৫, ২৯.৫-৩৯.৫ এবং ৩৯.৫-৪৯.৫। তৃতীয় শ্রেণির সংশোধিত শ্রেণি মধ্যক হল ৩৪.৫।",জেলা: দার্জিলিং\nশ্রেণী: একাদশ\nবিষয়: অর্থনীতি (Economics)\nপাঠ্য অধ্যায়: Adjustment of Class Limits in Frequency Distribution
1,1,True,,,4.0,4.0,4.0,4.0,4.0,4.0,True,,নবম শ্রেণির বিজ্ঞানের জন্য ত্বরণের উপর একটি অনুশীলনী প্রশ্ন দিন।,"প্রশ্ন: দেওলো পাহাড়ে একটি পরীক্ষার সময়, লাকপা একটি গাড়িকে ঢাল বেয়ে নামতে দেখল। গাড়িটির বেগ ৫.০ সেকেন্ডে ২.০ মিটার/সেকেন্ড থেকে বেড়ে ১২.০ মিটার/সেকেন্ড হয়। গাড়িটির ত্বরণ নির্ণয় কর। সমাধান: ধাপ ১: পর্যবেক্ষণ থেকে প্রদত্ত মানগুলি চিহ্নিত কর। প্রাথমিক বেগ (u) = ২.০ মিটার/সেকেন্ড, অন্তিম বেগ (v) = ১২.০ মিটার/সেকেন্ড, সময় (t) = ৫.০ সেকেন্ড। ধাপ ২: ত্বরণের সূত্রটি মনে কর: a = (v - u) / t। ধাপ ৩: সূত্রটিতে মানগুলি বসাও: a = (১২.০ - ২.০) / ৫.০। ধাপ ৪: লব গণনা কর: ১২.০ - ২.০ = ১০.০ মিটার/সেকেন্ড। ধাপ ৫: সময় দ্বারা ভাগ কর: ১০.০ / ৫.০ = ২.০ মিটার/সেকেন্ড²। চূড়ান্ত উত্তর: গাড়িটির ত্বরণ হল ২.০ মিটার/সেকেন্ড²।",জেলা: কালিম্পং\nশ্রেণী: ৯ম\nবিষয়: বিজ্ঞান (Science)\nপাঠ্য অধ্যায়: Acceleration
2,2,True,,,4.0,4.0,4.0,4.0,4.0,4.0,True,,চতুর্থ শ্রেণির গণিতের জন্য ট্যালিস বা ট্যালি চিহ্ন এবং ডেটা টেবিল বা তথ্য সারণি পড়ার ধারণাটি বুঝিয়ে দিন।,"ধারণার ব্যাখ্যা: চতুর্থ শ্রেণির গণিতে, আমাদের চারপাশের জিনিসপত্র গোনার সময় ট্যালি চিহ্ন খুব দ্রুত তথ্য গুনে নথিভুক্ত করতে সাহায্য করে। আমরা যখনই কোনো একটি জিনিস গুনি, তখন একটি লম্বালম্বি দাগ টানি। প্রতি পঞ্চম জিনিসটির জন্য, আমরা প্রথম চারটি দাগের ওপর একটি তির্যক দাগ টানি যাতে পাঁচটি দাগের একটি আঁটি তৈরি হয় (চারটি লম্বালম্বি দাগ এবং তার ওপর একটি কাটা দাগ)। ডেটা টেবিল বা তথ্য সারণি পড়া মানে সহজেই সংখ্যা খুঁজে পেতে এবং তুলনা করতে সারি ও কলামগুলো দেখা। কালিম্পংয়ের পাহাড়ের একটি স্কুলের কথা ভাবা যাক, যেখানে সোনম এবং লাকপা তাদের শিক্ষককে একটি সাংস্কৃতিক প্রদর্শনীর জন্য জিনিসপত্র সাজাতে সাহায্য করছে। তারা শিক্ষার্থীদের নিয়ে আসা বিভিন্ন স্থানীয় জিনিস গুনে দেখল: বড় এলাচ: ১২, আদা: ৮, ফুলের টব: ১৫, থাঙ্কা চিত্রশিল্প: ৫। প্রশ্ন: আমরা কীভাবে ট্যালি চিহ্নের সাহায্যে বড় এলাচের সংখ্যা প্রকাশ করব এবং তথ্য সারণিতে কোন জিনিসটির সংখ্যা সবচেয়ে বেশি? সমাধান: ধাপ ১: বড় এলাচের সংখ্যা হলো ১২। আমরা ট্যালি চিহ্নে ১২-কে পাঁচটি করে দুটি আঁটি এবং দুটি একক দাগ দিয়ে প্রকাশ করি: |||| |||| ||। ধাপ ২: তথ্য সারণির সমস্ত জিনিসগুলো দেখে (বড় এলাচ: ১২, আদা: ৮, ফুলের টব: ১৫, থাঙ্কা চিত্রশিল্প: ৫), আমরা সংখ্যাগুলোর তুলনা করি। ধাপ ৩: সবচেয়ে বেশি সংখ্যা হলো ১৫, যা ফুলের টবের ক্ষেত্রে রয়েছে। চূড়ান্ত উত্তর: বড় এলাচকে দুটি আঁটি ও দুটি একক দাগ দিয়ে ট্যালি করা হয় এবং

## 13. Summary


In [ ]:
score_columns = [
    "context_grounding",
    "faithfulness",
    "answer_relevancy",
    "instruction_following",
    "bengali_language_quality",
    "overall_score"
]

summary = final_df[score_columns].describe().T
display(summary)

print("Total rows:", len(final_df))
print("Passed:", int(final_df["passed"].fillna(False).sum()))
print("Failed / not passed:", int((~final_df["passed"].fillna(False)).sum()))

if "schema_valid" in final_df.columns:
    print("Schema valid:", int(final_df["schema_valid"].sum()))
    print("Schema invalid:", int((~final_df["schema_valid"]).sum()))

if "text_quality_issues" in final_df.columns:
    print(
        "Rows with pre-validation text issues:",
        int(final_df["text_quality_issues"].fillna("").astype(bool).sum())
    )


,count,mean,std,min,25%,50%,75%,max
context_grounding,1984.0,3.942036,0.296504,0.0,4.0,4.0,4.0,4.0
faithfulness,1984.0,3.964214,0.246467,0.0,4.0,4.0,4.0,4.0
answer_relevancy,1984.0,3.965726,0.241517,0.0,4.0,4.0,4.0,4.0
instruction_following,1984.0,3.944556,0.296135,0.0,4.0,4.0,4.0,4.0
bengali_language_quality,1984.0,3.878024,0.418003,0.0,4.0,4.0,4.0,4.0
overall_score,1984.0,3.938911,0.246198,0.0,4.0,4.0,4.0,4.0


Total rows: 1999
Passed: 1953
Failed / not passed: 46
Schema valid: 1999
Schema invalid: 0
Rows with pre-validation text issues: 24


## 14. Download final Excel report


In [ ]:
from google.colab import files

files.download(FINAL_XLSX)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>